# Phase 8.2 — Multiple Regression: Daily Crime Forecast per Community Area

**Goal:** predict the number of crimes in a community area on a given day (`crime_count`).

**Input:** `data/processed/area_day.parquet` — one row per community area per day (77 areas), including zero-crime days.
**Script:** `ml/train_regression.py` (this notebook reads its saved results; it does not retrain).

## 1. Method

- **Table:** 77 areas × 1,348 days (2023-01-01 to 2026-09-09) = 103,796 rows; the first 30 days of each area are dropped because they have no 30-day history, leaving 101,486 rows. Days with no recorded crime are kept as `crime_count = 0` (about 4.2% of area-days).
- **Time-based split (never random):** train = 2023–2024, test = 2025 onwards.
- **History features** use only *past* days: `lag_1`, `lag_7`, `lag_30` (counts 1, 7, 30 days earlier) and rolling means that end *yesterday*. Today's count is never used to predict today.
- **Three models are compared on the test period:**
  1. Naive baseline: same count as one week earlier (`lag_7`).
  2. Linear regression with history + calendar features (day of week, month, weekend, holiday).
  3. The same model plus weather (temperature, rainfall) and census features (poverty rate, per-capita income, population).
- **Metrics:** MAE, RMSE and R² on the test set.

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

BASE = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
RESULTS, MODELS, SHOTS = f"{BASE}/ml/results", f"{BASE}/ml/models", f"{BASE}/docs/screenshots"
pd.set_option("display.width", 200)

## 2. Input table

In [ ]:
d = pd.read_parquet(f"{BASE}/data/processed/area_day.parquet")
d["date"] = pd.to_datetime(d["date"])
print("Shape:", d.shape, "| areas:", d["community_area"].nunique())
print("Missing cells:", int(d.isna().sum().sum()))
print("Zero-crime area-days:", int((d["crime_count"] == 0).sum()), f"({(d['crime_count'] == 0).mean()*100:.1f}%)")
print(d.groupby("split")["date"].agg(["min", "max", "count"]))
display(d["crime_count"].describe().round(2).to_frame().T)

## 3. Multicollinearity check (VIF)

In [ ]:
before = pd.read_csv(f"{RESULTS}/regression_vif_before.csv")
after = pd.read_csv(f"{RESULTS}/regression_vif.csv")
vif = before.merge(after, on="feature", how="left", suffixes=("_before", "_after"))
display(vif.round(2))
print("Highest VIF before:", before.iloc[0]["feature"], round(before.iloc[0]["vif"], 1))
print("Highest VIF after :", after.iloc[0]["feature"], round(after.iloc[0]["vif"], 1))

The history features overlap by construction: `rolling_7` covers the same days as `lag_1`…`lag_7`, and `rolling_30` overlaps all the lags. VIF was 43.7 for `rolling_7` and 41.9 for `rolling_30`. Four variants were tested; dropping `rolling_7` lowered the highest VIF to 17.5 with **no change in test error** (MAE 2.4302 → 2.4308), while dropping `rolling_30` instead raised MAE to 2.51 and dropping both raised it to 2.67. So `rolling_7` was removed and `rolling_30` kept. The poverty/income correlation was −0.57, below the 0.8 limit, so both stay.

## 4. Model comparison (test period: 2025 onwards)

In [ ]:
m = pd.read_csv(f"{RESULTS}/regression_metrics.csv")
display(m.round(4))
naive, base, enh = m.iloc[0], m.iloc[1], m.iloc[2]
print(f"Lag+calendar vs naive      : MAE {naive['MAE']:.3f} -> {base['MAE']:.3f} ({(1-base['MAE']/naive['MAE'])*100:.1f}% lower)")
print(f"+census+weather vs lag+cal : MAE {base['MAE']:.4f} -> {enh['MAE']:.4f} ({(1-enh['MAE']/base['MAE'])*100:.2f}% lower)")

In [ ]:
display(Image(f"{SHOTS}/12_regression_actual_vs_predicted.png"))

## 5. Coefficients (enhanced model)

In [ ]:
coef = pd.read_csv(f"{RESULTS}/regression_coefficients.csv")
display(coef.head(12).round(3))

Numeric inputs were standardised, so their coefficients are comparable. `rolling_30` (the recent 30-day level of the area) is by far the largest. Month terms are relative to the reference month (January) after the history features are taken into account.

## 6. Using the saved model (what the API does)

In [ ]:
store = joblib.load(f"{MODELS}/regression.pkl")
print("Dropped for collinearity:", store["dropped_for_collinearity"])
print("Features (enhanced):", store["features_enhanced"])

test = d[d["split"] == "test"].copy()
test["pred"] = np.clip(store["enhanced"].predict(test[store["features_enhanced"]]), 0, None)

# Re-check the headline numbers from the saved model
mae = (test["crime_count"] - test["pred"]).abs().mean()
print(f"Recomputed test MAE: {mae:.4f}")

# One example prediction
row = test.iloc[-1]
print(f"Example: area {int(row['community_area'])} on {row['date'].date()} -> predicted {row['pred']:.1f}, actual {int(row['crime_count'])}")

# Where is the model least accurate?
by_area = (test.assign(err=(test["crime_count"] - test["pred"]).abs())
               .groupby("community_area")["err"].mean().sort_values(ascending=False))
print("\nAreas with the largest average error (MAE):")
print(by_area.head(5).round(2).to_string())

## 7. Interpretation and limitations

- **The model clearly beats the naive baseline.** Test MAE falls from 3.31 to 2.43 crimes per area per day and R² rises from 0.63 to 0.80.
- **Census and weather add very little:** MAE improves by only about 0.007 (0.28%). Census values are constant within an area, and the history features already carry each area's typical level; seasonal effects are largely captured by the month. This is a real finding and is reported as is, not tuned away.
- **What the scores mean:** they measure a *next-day* forecast from the actual counts of the previous days. A 7-day forecast must feed its own predictions back in as lags and will be less accurate.
- **Predictions are clipped at 0** because counts cannot be negative. Linear regression can still miss sudden spikes.
- **Not causal:** the coefficients describe associations in recorded crime. Recorded crime depends on reporting and policing as well as on actual crime.
- **Data note:** the last day of the original file (10 Sep 2026) held only 12 midnight records and was removed, so the data ends on 9 Sep 2026.